In [1]:
# -----------------------------
# 01_dataset_eda: Persistent Drive split writer (Paste & run at top of 01_dataset_eda.ipynb)
# -----------------------------
import os
from pathlib import Path
import re
import json
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import Counter
import shutil

# ---------------- CONFIG: change only if you must ----------------
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
# Exact raw data folders you listed earlier (do not change unless needed)
RAW_DATA_ROOTS = {
    "English": Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/English"),
    "Hindi":   Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/Hindi"),
    "Marathi": Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/Marathi"),
}
# Where to persist splits and outputs on Drive
WORK_ROOT = DRIVE_PROJECT_ROOT.resolve()
SPLITS_ROOT = WORK_ROOT / "data_splits"        # persistent splits path (will be created)
OUTPUTS_ROOT = WORK_ROOT / "outputs"           # base for outputs (models, baselines etc.)
# canonical ordering you required
CANONICAL_LABEL_ORDER = {
    "English": ["G", "PG", "PG-13", "R", "NC-17"],
    "Hindi":   ["U", "UA", "A"],
    "Marathi": ["U", "UA"]
}
# split ratios and seed
TRAIN_PCT, VAL_PCT, TEST_PCT = 0.70, 0.15, 0.15
RANDOM_STATE = 42
METADATA = {"chunk_max_len": 256, "chunk_stride": 64}
# -----------------------------------------------------------------

# Mount drive if needed (idempotent)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("[OK] Drive mounted at /content/drive")
except Exception as e:
    print("[INFO] Drive mount skipped or not running in Colab:", e)

# Ensure working directories exist
for p in [WORK_ROOT, SPLITS_ROOT, OUTPUTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
print(f"[OK] Using WORK_ROOT = {WORK_ROOT}")
print(f"[OK] SPLITS_ROOT = {SPLITS_ROOT}")

# filename parsing helper
LABEL_TOKEN_PATTERN = re.compile(r'^(?P<label>G|PG-13|PG|NC-17|R|U|UA|A)(?:[_\- ]+)(?P<title>.+?)(?:[_\- ]+(?P<year>\d{4}))?(?:\.\w+)?$', flags=re.IGNORECASE)

def parse_filename(fname):
    name = Path(fname).name
    stem = os.path.splitext(name)[0]
    m = LABEL_TOKEN_PATTERN.match(stem)
    if m:
        lab = m.group("label").upper().replace("PG13","PG-13")
        title = m.group("title").replace('_',' ').replace('-',' ').strip()
        yr = m.group("year")
        year = int(yr) if yr and yr.isdigit() else None
        return {"label": lab, "title": title, "year": year}
    # fallback search for any known label token
    tokens = re.split(r'[_\-\s]+', stem)
    candidates = sorted(set(sum([v for v in CANONICAL_LABEL_ORDER.values()], [])), key=lambda x: -len(x))
    if tokens:
        first = tokens[0].upper()
        for c in candidates:
            if first == c or first.startswith(c):
                title = " ".join(tokens[1:]) if len(tokens) > 1 else ""
                yr = tokens[-1] if tokens[-1].isdigit() and len(tokens[-1])==4 else None
                year = int(yr) if yr else None
                return {"label": c, "title": title, "year": year}
    return None

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

# Iterate languages and build splits
summary = {}
for lang, raw_root in RAW_DATA_ROOTS.items():
    print(f"\n[LANG] {lang}  raw_root={raw_root}")
    if not raw_root.exists():
        raise FileNotFoundError(f"Raw dataset folder not found for {lang} at {raw_root}. Please verify.")
    # collect files with common text-like extensions
    files = sorted([p for p in raw_root.glob("*") if p.is_file() and p.suffix.lower() in {'.txt', '.srt', '.md'}])
    rows = []
    for fp in files:
        parsed = parse_filename(fp.name)
        if parsed is None:
            # attempt to find label token anywhere in filename
            lab_found = None
            for lab in sum(CANONICAL_LABEL_ORDER.values(), []):
                if re.search(r'\b' + re.escape(lab) + r'\b', fp.name, flags=re.IGNORECASE):
                    lab_found = lab; break
            if lab_found is None:
                print(f"[WARN] Could not parse label/title/year from filename: {fp.name} -> skipping")
                continue
            parsed = {"label": lab_found, "title": fp.stem, "year": None}
        label = parsed['label'].upper()
        if label.replace(' ','') == 'PG13':
            label = 'PG-13'
        rows.append({
            "file_path": str(fp.resolve()),
            "filename": fp.name,
            "label": label,
            "title": parsed.get("title") or fp.stem,
            "year": parsed.get("year"),
            "language": lang
        })
    if not rows:
        print(f"[WARN] No usable files detected for {lang}. Continuing.")
        continue

    df = pd.DataFrame(rows)
    # create splits dir for this language (persistent Drive)
    lang_dir = SPLITS_ROOT / lang
    ensure_dir(lang_dir)

    # stratified split where possible
    label_counts = df['label'].value_counts().to_dict()
    small_labels = [l for l,c in label_counts.items() if c < 2]
    if small_labels:
        print(f"[WARN] Some labels small for {lang}: {small_labels} -> using random split")
        df_train, df_temp = train_test_split(df, test_size=(1-TRAIN_PCT), random_state=RANDOM_STATE, shuffle=True)
        val_ratio = VAL_PCT / (VAL_PCT + TEST_PCT)
        df_val, df_test = train_test_split(df_temp, test_size=(1-val_ratio), random_state=RANDOM_STATE, shuffle=True)
    else:
        df_train, df_temp = train_test_split(df, test_size=(1-TRAIN_PCT), random_state=RANDOM_STATE, shuffle=True, stratify=df['label'])
        val_ratio = VAL_PCT / (VAL_PCT + TEST_PCT)
        df_val, df_test = train_test_split(df_temp, test_size=(1-val_ratio), random_state=RANDOM_STATE, shuffle=True, stratify=df_temp['label'])

    # reset index and save CSVs (overwrite existing in Drive)
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    for name, dff in [("train.csv", df_train), ("val.csv", df_val), ("test.csv", df_test)]:
        dest = lang_dir / name
        dff.to_csv(dest, index=False, encoding="utf8")
        print(f"[OK] Wrote {dest} ({len(dff)})")

    # build canonical label_map: present canonical labels first, then extras
    uniq_labels = set(df['label'].unique())
    label_map = {}
    idx = 0
    for lab in CANONICAL_LABEL_ORDER.get(lang, []):
        if lab in uniq_labels:
            label_map[lab] = idx; idx += 1
    extras = sorted([l for l in uniq_labels if l not in label_map])
    for lab in extras:
        label_map[lab] = idx; idx += 1
    inv_label_map = {str(v):k for k,v in label_map.items()}
    label_map_json = {"label_map": label_map, "inv_label_map": inv_label_map, "metadata": METADATA}
    with open(lang_dir / "label_map.json", "w", encoding="utf8") as f:
        json.dump(label_map_json, f, indent=2, ensure_ascii=False)
    # combined manifest
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
    df_all.to_csv(lang_dir / "all_files_manifest.csv", index=False, encoding="utf8")

    summary[lang] = {"total": len(df_all), "per_label_counts": dict(df_all['label'].value_counts())}

print("\n=== SUMMARY ===")
for lang, s in summary.items():
    print(f"- {lang}: total={s['total']} labels={s['per_label_counts']}")
print(f"[DONE] All splits and label_map.json saved under: {SPLITS_ROOT}")


Mounted at /content/drive
[OK] Drive mounted at /content/drive
[OK] Using WORK_ROOT = /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2
[OK] SPLITS_ROOT = /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits

[LANG] English  raw_root=/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/English
[OK] Wrote /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits/English/train.csv (799)
[OK] Wrote /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits/English/val.csv (171)
[OK] Wrote /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits/English/test.csv (172)

[LANG] Hindi  raw_root=/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/Hindi
[OK] Wrote /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits/Hindi/train.csv (142)
[OK] Wrote /content/drive/MyDrive/Ph